# Albion gathering bot - the big model

The first model was trained on one dataset of 657 pictures, all shot in the same bright
grassy place. It scores 0.74 on a tree there and 0.26 on a tree in Forgotten Woods,
which is why the bot goes blind the moment you walk into a dark forest.

This notebook fixes that by training on **every public Albion dataset at once**, around
six thousand pictures from a dozen different people, their setups, their zones and their
times of day. Variety is the whole point: a model that has seen trees in twenty places
recognises the twenty-first.

The datasets all name things differently, one says `rough logs`, another says `T4_TREE`,
another just `wood`. Step 4 rewrites every label onto the five names the bot actually
uses, so they train as one.

**Do this first:** menu **Runtime -> Change runtime type -> T4 GPU -> Save**.

## Step 1 - check the GPU

There is a lot more data this time, so a GPU is not optional. Expect `Tesla T4`.

In [ ]:
!nvidia-smi

## Step 2 - install yolov5

In [ ]:
%cd /content
!git clone https://github.com/ultralytics/yolov5.git
%cd /content/yolov5

# The version the bot on your PC runs, so the weights load there without surprises.
!git checkout --quiet f9e040b2c57e1db73f138ea420494fdeb05939da

!pip install --quiet -r requirements.txt roboflow

# wandb stops to ask for an account halfway through the training, get rid of it first.
!pip uninstall --quiet -y wandb
%env WANDB_MODE=disabled
%env WANDB_SILENT=true

import torch
print()
print("torch", torch.__version__)
print("GPU  ", torch.cuda.get_device_name(0) if torch.cuda.is_available()
       else "NONE - set Runtime > Change runtime type > T4 GPU")

## Step 3 - download every dataset

Same Roboflow key as last time, from <https://app.roboflow.com/settings/api>.

One key downloads all of them, they are public. If one is unavailable it is skipped and
the rest still go ahead, so a single broken dataset does not waste the run.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

API_KEY = getpass("Paste your Roboflow private API key, then press Enter: ")
rf = Roboflow(api_key=API_KEY)

# (workspace, project, short name used to prefix the files). Taken from the Universe
# urls: universe.roboflow.com/<workspace>/<project>
SOURCES = [
    ("albion-online",            "ao-resource-detection",  "aores"),
    ("albiononline-6no0l",       "albion-bpgjq",           "albion2"),
    ("joao-angelo",              "albion-resources-6myx5", "joao"),
    ("ai-workspace-yolo-ghcsl",  "albion-online---farm-bot", "farmbot"),
    ("albion-zmhvd",             "ore1",                   "ore1"),
    ("albiononlinez",            "albion-enemys",          "enemys"),
    ("albiononline-c8fxi",       "albiongathering",        "original"),
]

downloaded = []

for workspace, name, prefix in SOURCES:
    try:
        project = rf.workspace(workspace).project(name)

        # The newest version, whatever number it happens to be.
        versions = sorted(project.versions(), key=lambda v: int(str(v.version).split("/")[-1]))
        dataset = versions[-1].download("yolov5")

        downloaded.append((prefix, dataset.location))
        print(f"OK    {prefix:9} {dataset.location}")
    except Exception as e:
        print(f"SKIP  {prefix:9} {type(e).__name__}: {str(e)[:110]}")

print(f"\n{len(downloaded)} of {len(SOURCES)} datasets downloaded")

## Step 4 - put every label onto the same five names

Each dataset invented its own class names. This resolves all of them onto the names the
bot understands, using the very same word matching that `Application/Albion/resources.py`
does, so whatever trains here is what the bot recognises later.

A class that matches nothing, a chest or a player name, has its boxes dropped. The
picture is still kept: an image with no boxes teaches the model what is *not* a resource,
which is exactly what stops it clicking on scenery.

Read the printed mapping. Anything landing on `dropped` that looks like a real resource
is a word missing from the alias lists, and worth adding on both sides.

In [ ]:
import re
import shutil
from collections import Counter
from pathlib import Path

import yaml

# The same aliases as Application/Albion/resources.py, in the same order, plus monsters,
# which the bot does not gather but does walk around.
ALIASES = {
    # "rough" is deliberately absent: tier 2 wood is "Rough Logs" and tier 2 rock is
    # "Rough Stone", and tree being tried first claimed the rock. "logs" catches the wood.
    "tree":    ("tree", "trees", "wood", "log", "logs", "birch", "chestnut",
                "pine", "cedar", "bloodoak", "ashenbark"),
    "stone":   ("stone", "stones", "rock", "rocks", "limestone", "sandstone", "travertine",
                "granite", "slate", "basalt", "marble"),
    "ore":     ("ore", "ores", "copper", "tin", "iron", "titanium", "runite", "meteorite",
                "adamantium"),
    "fiber":   ("fiber", "fibre", "cotton", "flax", "hemp", "amberleaf", "sunflax", "ghost",
                "redleaf", "silk"),
    "hide":    ("hide", "hides", "animal", "rabbit", "fox", "boar", "wolf", "bear", "deer"),
    "monster": ("monster", "monsters", "mob", "mobs", "enemy", "enemies", "creature"),
}

CANONICAL = list(ALIASES)


def resolve(label):
    """Find which of the five a dataset's class name means, None when it means none."""
    label = label.lower().strip()
    words = re.split(r"[^a-z0-9]+|(?<=[a-z])(?=[0-9])|(?<=[0-9])(?=[a-z])", label)

    for canonical, aliases in ALIASES.items():
        for alias in aliases:
            if alias in words or (len(alias) >= 4 and alias in label.replace(" ", "")):
                return canonical

    return None


merged = Path("/content/albion_merged")
shutil.rmtree(merged, ignore_errors=True)

for split in ("train", "valid", "test"):
    (merged / split / "images").mkdir(parents=True, exist_ok=True)
    (merged / split / "labels").mkdir(parents=True, exist_ok=True)

instances = Counter()
pictures = Counter()

for prefix, location in downloaded:
    location = Path(location)
    config = yaml.safe_load((location / "data.yaml").read_text())

    names = config["names"]
    names = list(names.values()) if isinstance(names, dict) else names

    mapping = {}
    print(f"\n{prefix}")

    for index, name in enumerate(names):
        canonical = resolve(name)
        mapping[index] = CANONICAL.index(canonical) if canonical else None
        print(f"   {name!r} -> {canonical or 'dropped'}")

    for split in ("train", "valid", "test"):
        images = location / split / "images"
        labels = location / split / "labels"

        if not images.exists():
            continue

        for picture in images.iterdir():
            if picture.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue

            kept = []
            label = labels / f"{picture.stem}.txt"

            if label.exists():
                for line in label.read_text().splitlines():
                    parts = line.split()

                    if not parts:
                        continue

                    new = mapping.get(int(parts[0]))

                    if new is None:
                        continue

                    kept.append(" ".join([str(new)] + parts[1:]))
                    instances[CANONICAL[new]] += 1

            # The name is prefixed because two datasets happily use the same file names.
            stem = f"{prefix}_{picture.stem}"
            shutil.copy(picture, merged / split / "images" / f"{stem}{picture.suffix}")
            (merged / split / "labels" / f"{stem}.txt").write_text("\n".join(kept))
            pictures[split] += 1

print("\n" + "=" * 50)
print("pictures :", dict(pictures), "total", sum(pictures.values()))
print("boxes    :", dict(instances), "total", sum(instances.values()))

## Step 5 - write the config and sanity check it

If a class has very few boxes the model will be poor at it. Anything under a few hundred
is worth knowing about before spending an hour training.

In [ ]:
data_yaml = merged / "data.yaml"

config = {
    "train": str(merged / "train" / "images"),
    "val": str(merged / "valid" / "images"),
    "nc": len(CANONICAL),
    "names": CANONICAL,
}

if pictures.get("test"):
    config["test"] = str(merged / "test" / "images")

data_yaml.write_text(yaml.dump(config))

print(data_yaml.read_text())

for name in CANONICAL:
    count = instances.get(name, 0)
    note = "plenty" if count >= 500 else "thin, expect misses" if count >= 100 else "TOO FEW"
    print(f"  {name:8} {count:6} boxes   {note}")

## Step 6 - train

Budget **four to eight hours** on a T4 for the full 150 epochs, though `--patience 30`
usually ends it well before that. Check the time of the first epoch and multiply.

`--cache disk` rather than the RAM cache on purpose: eight thousand pictures at 640 is
around 7.5GB, and a free Colab box has about 12.5GB for everything, so caching them in
memory gets the process killed before the first epoch. Disk is a little slower per epoch
and does not fall over. If it still dies while caching, drop the flag altogether.

`--exist-ok` keeps the run in `runs/train/albion_merged` instead of making
`albion_merged2`, `albion_merged3` and so on, which the steps below read from by name.

**You are never all or nothing:** `best.pt` is written after every epoch, so stopping the
cell when `mAP50` flattens leaves you a model you can use.

Leave the tab open. Colab throws the runtime away if you disconnect.

`yolov5s` is kept on purpose: it is the small fast model, and the bot runs it on your CPU
a couple of times a second. `yolov5m` would score a little higher and run half as fast.

In [ ]:
!python train.py \
  --img 640 \
  --batch 32 \
  --epochs 150 \
  --patience 30 \
  --data {data_yaml} \
  --weights yolov5s.pt \
  --name albion_merged \
  --exist-ok \
  --cache disk \
  --workers 4

## Step 7 - did it learn

`mAP50` is the number that matters. Over `0.8` is a bot that finds things reliably.

Compare the confusion matrix with the old one: the diagonal should be stronger for trees,
which was the weak class at 0.77 before.

In [ ]:
from IPython.display import Image, display

run = "/content/yolov5/runs/train/albion_merged"

display(Image(f"{run}/results.png", width=1000))
display(Image(f"{run}/confusion_matrix.png", width=700))

## Step 8 - try it on pictures from your own game

This is the real test. Upload a few frames from `images\dataset` on your PC, the ones
the bot collected in Forgotten Woods, and see whether the new model finds trees in them.
The old one scored 0.26 there.

Run the cell, press **Choose Files**, pick a handful.

In [ ]:
from pathlib import Path

from google.colab import files
from IPython.display import Image, display

mine = Path("/content/mine")
mine.mkdir(exist_ok=True)

for name in files.upload():
    Path(name).rename(mine / name)

!python detect.py --weights {run}/weights/best.pt --img 640 --conf 0.4 \
    --source /content/mine --name mine --exist-ok

for picture in sorted(Path("/content/yolov5/runs/detect/mine").iterdir())[:6]:
    display(Image(str(picture), width=700))

## Step 9 - download best.pt

Goes to your Downloads folder. Tell Claude once it lands and it gets installed.

Keep the old one, it is still better in the bright zone it was trained on, and comparing
the two on the same picture is the honest way to tell whether this was worth it.

In [ ]:
import shutil

from google.colab import files

shutil.copy(f"{run}/weights/best.pt", "/content/best_merged.pt")

size = round(shutil.os.path.getsize("/content/best_merged.pt") / 1e6, 1)
print(f"best_merged.pt is {size} MB, downloading...")

files.download("/content/best_merged.pt")